# 02 — GNN Training & Evaluation

This notebook trains a **GraphSAGE** model on the EU AI Act knowledge graph using **self-supervised contrastive learning** (InfoNCE loss).

## Contents
1. Setup & Load Graph
2. Data Inspection
3. Edge Splitting (Train / Val / Test)
4. Model Initialisation
5. Training Loop
6. Training Curves
7. Embedding Analysis
8. Nearest-Neighbour Retrieval Demo
9. Baseline Comparison
10. Save Final Embeddings

---
## 1. Setup & Load Graph

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
import yaml

from src.gnn.models import get_model, GraphSAGEEncoder
from src.gnn.train import (
    load_config, set_seed, split_edges,
    sample_contrastive_pairs, infonce_loss,
)

# Style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 5)

# Config
config = load_config('../configs/config.yaml')

# Seed
set_seed(config['project']['seed'])

# Device
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f'Device: {device}')
print('Setup complete ✓')

In [ ]:
# Load graph
graph_path = f"../{config['paths']['graph_object']}"
data = torch.load(graph_path, weights_only=False)

# Load node metadata for analysis
nodes_df = pd.read_csv(f"../{config['paths']['nodes_csv']}")
edges_df = pd.read_csv(f"../{config['paths']['edges_csv']}")

print(f'Graph loaded ✓')
print(f'  Nodes:      {data.num_nodes}')
print(f'  Edges:      {data.num_edges}')
print(f'  Features:   {data.x.shape}')
print(f'  Edge types:  {data.edge_type_mapping}')

---
## 2. Data Inspection

In [ ]:
# Feature statistics
x_np = data.x.numpy()
print(f'Feature matrix shape: {x_np.shape}')
print(f'Feature range:        [{x_np.min():.4f}, {x_np.max():.4f}]')
print(f'Feature mean:         {x_np.mean():.4f}')
print(f'Feature std:          {x_np.std():.4f}')
print(f'Any NaNs:             {np.isnan(x_np).any()}')
print(f'Any Infs:             {np.isinf(x_np).any()}')

# Edge type distribution
print(f'\nEdge type distribution:')
edge_types_tensor = data.edge_type
inv_type_map = {v: k for k, v in data.edge_type_mapping.items()}
for idx in sorted(inv_type_map.keys()):
    count = (edge_types_tensor == idx).sum().item()
    print(f'  {inv_type_map[idx]:15s} → {count}')

In [ ]:
# Degree distribution of the PyG graph
from torch_geometric.utils import degree

deg = degree(data.edge_index[0], num_nodes=data.num_nodes)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(deg.numpy(), bins=40, color='steelblue', edgecolor='white', alpha=0.8)
ax.set_xlabel('Degree')
ax.set_ylabel('Count')
ax.set_title(f'Node Degree Distribution (mean={deg.mean():.1f}, max={deg.max():.0f})')
ax.axvline(deg.mean(), color='red', linestyle='--', alpha=0.6, label=f'Mean ({deg.mean():.1f})')
ax.legend()
plt.tight_layout()
plt.show()

---
## 3. Edge Splitting (Train / Val / Test)

In [ ]:
# Split edges
edge_splits = split_edges(
    data.edge_index,
    train_ratio=config['training']['train_ratio'],
    val_ratio=config['training']['val_ratio'],
)

print('Edge split:')
for split, ei in edge_splits.items():
    print(f'  {split:6s}: {ei.size(1):5d} edges ({ei.size(1)/data.num_edges*100:.1f}%)')

---
## 4. Model Initialisation

In [ ]:
# Build model
in_channels = data.x.size(1)
model = get_model(config['gnn'], in_channels).to(device)

print(f'Architecture:    {config["gnn"]["architecture"]}')
print(f'Input dim:       {in_channels}')
print(f'Hidden dim:      {config["gnn"]["hidden_channels"]}')
print(f'Output dim:      {config["gnn"]["out_channels"]}')
print(f'Num layers:      {config["gnn"]["num_layers"]}')
print(f'Dropout:         {config["gnn"]["dropout"]}')
print(f'Aggregation:     {config["gnn"].get("aggregation", "mean")}')
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'\nModel:\n{model}')

---
## 5. Training Loop

**Self-supervised contrastive learning** using InfoNCE loss:
- **Positive pairs**: connected nodes (edges in the graph)
- **Negative pairs**: randomly sampled unconnected nodes
- **Objective**: push connected node embeddings closer, unconnected further apart

In [ ]:
# Training hyperparameters
EPOCHS = config['training']['epochs']
LR = config['training']['learning_rate']
WEIGHT_DECAY = config['training']['weight_decay']
NUM_NEGATIVES = config['training']['num_negatives']
TEMPERATURE = config['training']['temperature']
PATIENCE = config['training']['patience']

print('Training Configuration:')
print(f'  Epochs:          {EPOCHS}')
print(f'  Learning rate:   {LR}')
print(f'  Weight decay:    {WEIGHT_DECAY}')
print(f'  Num negatives:   {NUM_NEGATIVES}')
print(f'  Temperature:     {TEMPERATURE}')
print(f'  Patience:        {PATIENCE}')
print(f'  Scheduler:       {config["training"].get("scheduler", "none")}')

In [ ]:
# Optimizer & Scheduler
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

optimizer = Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

scheduler = None
if config['training'].get('scheduler') == 'cosine':
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    print('Using CosineAnnealingLR scheduler')
else:
    print('No scheduler')

In [ ]:
# ── Training loop ──

train_losses = []
val_losses = []
lr_history = []

best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

# Move data to device
x = data.x.to(device)
train_edges = edge_splits['train'].to(device)
val_edges = edge_splits['val'].to(device)

# Checkpoint path
ckpt_dir = Path(f"../{config['paths']['model_checkpoint']}").parent
ckpt_dir.mkdir(parents=True, exist_ok=True)
ckpt_path = f"../{config['paths']['model_checkpoint']}"

print(f'Starting training for up to {EPOCHS} epochs...\n')

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    embeddings = model(x, train_edges)

    anchors, positives, negatives = sample_contrastive_pairs(
        edge_splits['train'], data.num_nodes, NUM_NEGATIVES
    )

    anchor_emb = embeddings[anchors.to(device)]
    positive_emb = embeddings[positives.to(device)]
    negative_emb = embeddings[negatives.to(device)]

    loss = infonce_loss(anchor_emb, positive_emb, negative_emb, TEMPERATURE)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_loss = loss.item()
    train_losses.append(train_loss)

    # ── Validate ──
    model.eval()
    with torch.no_grad():
        embeddings_val = model(x, train_edges)
        a_val, p_val, n_val = sample_contrastive_pairs(
            edge_splits['val'], data.num_nodes, NUM_NEGATIVES
        )
        val_loss = infonce_loss(
            embeddings_val[a_val.to(device)],
            embeddings_val[p_val.to(device)],
            embeddings_val[n_val.to(device)],
            TEMPERATURE
        ).item()
    val_losses.append(val_loss)

    # LR tracking
    current_lr = optimizer.param_groups[0]['lr']
    lr_history.append(current_lr)

    if scheduler:
        scheduler.step()

    # ── Early stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_epoch = epoch
        torch.save(model.state_dict(), ckpt_path)
    else:
        patience_counter += 1

    # ── Logging ──
    if epoch % 10 == 0 or epoch == 1:
        marker = ' ← best' if epoch == best_epoch else ''
        print(
            f'Epoch {epoch:4d}/{EPOCHS} | '
            f'Train Loss: {train_loss:.4f} | '
            f'Val Loss: {val_loss:.4f} | '
            f'LR: {current_lr:.6f}{marker}'
        )

    if patience_counter >= PATIENCE:
        print(f'\n⏹ Early stopping at epoch {epoch} (patience={PATIENCE})')
        break

print(f'\n✓ Training complete. Best val loss: {best_val_loss:.4f} at epoch {best_epoch}')
print(f'  Checkpoint saved to: {ckpt_path}')

---
## 6. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
epochs_range = range(1, len(train_losses) + 1)
axes[0].plot(epochs_range, train_losses, label='Train Loss', color='#2196F3', alpha=0.8)
axes[0].plot(epochs_range, val_losses, label='Val Loss', color='#FF9800', alpha=0.8)
axes[0].axvline(best_epoch, color='green', linestyle='--', alpha=0.5, label=f'Best (epoch {best_epoch})')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('InfoNCE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

# Learning rate schedule
axes[1].plot(epochs_range, lr_history, color='#4CAF50')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Learning Rate')
axes[1].set_title('Learning Rate Schedule')

plt.tight_layout()
plt.show()

# Final stats
print(f'Final train loss:  {train_losses[-1]:.4f}')
print(f'Final val loss:    {val_losses[-1]:.4f}')
print(f'Best val loss:     {best_val_loss:.4f} (epoch {best_epoch})')
print(f'Total epochs:      {len(train_losses)}')

---
## 7. Embedding Analysis

Load the best model and generate final embeddings, then visualise with t-SNE.

In [ ]:
# Load best model
model.load_state_dict(torch.load(ckpt_path, weights_only=True, map_location=device))
model.eval()

# Generate final embeddings
with torch.no_grad():
    all_edges = data.edge_index.to(device)
    gnn_embeddings = model(x, all_edges).cpu().numpy()

print(f'GNN embeddings shape: {gnn_embeddings.shape}')
print(f'L2 norm (should be ~1.0): {np.linalg.norm(gnn_embeddings, axis=1).mean():.4f}')

In [ ]:
# t-SNE visualisation coloured by node type
print('Running t-SNE (this may take a moment)...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000)
emb_2d = tsne.fit_transform(gnn_embeddings)

# Colour map
type_list = nodes_df['type'].values
unique_types = sorted(set(type_list))
color_map = {
    'article': '#E53935',
    'paragraph': '#1E88E5',
    'recital': '#43A047',
    'definition': '#FB8C00',
    'annex': '#8E24AA',
    'annex_item': '#00ACC1',
}

fig, ax = plt.subplots(figsize=(12, 9))
for ntype in unique_types:
    mask = type_list == ntype
    ax.scatter(
        emb_2d[mask, 0], emb_2d[mask, 1],
        c=color_map.get(ntype, '#999'),
        label=f'{ntype} ({mask.sum()})',
        s=15, alpha=0.7
    )

ax.legend(fontsize=9, loc='upper right')
ax.set_title('t-SNE of GNN Node Embeddings (coloured by type)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# t-SNE coloured by chapter (articles & paragraphs only)
chapter_mask = nodes_df['type'].isin(['article', 'paragraph'])
chapters = nodes_df.loc[chapter_mask, 'chapter'].values
emb_2d_ch = emb_2d[chapter_mask]

chapter_colors = {
    'I': '#E53935', 'II': '#D81B60', 'III': '#8E24AA', 'IV': '#5E35B1',
    'V': '#3949AB', 'VI': '#1E88E5', 'VII': '#039BE5', 'VIII': '#00ACC1',
    'IX': '#00897B', 'X': '#43A047', 'XI': '#7CB342', 'XII': '#FDD835',
    'XIII': '#FB8C00'
}

fig, ax = plt.subplots(figsize=(12, 9))
for ch in sorted(chapter_colors.keys(), key=lambda c: list(chapter_colors.keys()).index(c)):
    mask = chapters == ch
    if mask.sum() > 0:
        ax.scatter(
            emb_2d_ch[mask, 0], emb_2d_ch[mask, 1],
            c=chapter_colors[ch], label=f'Ch. {ch} ({mask.sum()})',
            s=20, alpha=0.75
        )

ax.legend(fontsize=8, loc='upper right', ncol=2)
ax.set_title('t-SNE of GNN Embeddings — Articles & Paragraphs by Chapter', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Embedding similarity within vs across node types
intra_sims = []
inter_sims = []

np.random.seed(42)
sample_size = 200
idx_sample = np.random.choice(len(gnn_embeddings), min(sample_size, len(gnn_embeddings)), replace=False)

emb_sample = gnn_embeddings[idx_sample]
types_sample = type_list[idx_sample]
sim_matrix = cosine_similarity(emb_sample)

for i in range(len(idx_sample)):
    for j in range(i + 1, len(idx_sample)):
        if types_sample[i] == types_sample[j]:
            intra_sims.append(sim_matrix[i, j])
        else:
            inter_sims.append(sim_matrix[i, j])

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(intra_sims, bins=50, alpha=0.6, label=f'Intra-type (n={len(intra_sims)})', color='#4CAF50', density=True)
ax.hist(inter_sims, bins=50, alpha=0.6, label=f'Inter-type (n={len(inter_sims)})', color='#FF9800', density=True)
ax.set_xlabel('Cosine Similarity')
ax.set_ylabel('Density')
ax.set_title('Intra-type vs Inter-type Embedding Similarity')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Mean intra-type similarity: {np.mean(intra_sims):.4f}')
print(f'Mean inter-type similarity: {np.mean(inter_sims):.4f}')
print(f'Difference:                 {np.mean(intra_sims) - np.mean(inter_sims):.4f}')

---
## 8. Nearest-Neighbour Retrieval Demo

Show the top-k most similar nodes for selected query nodes to sanity-check the embeddings.

In [ ]:
def find_nearest_neighbours(query_node_id, embeddings, nodes_df, top_k=10):
    """Find most similar nodes to a given query node."""
    node_ids = nodes_df['node_id'].values
    idx = np.where(node_ids == query_node_id)[0][0]

    query_emb = embeddings[idx:idx+1]
    sims = cosine_similarity(query_emb, embeddings)[0]

    # Exclude self
    sims[idx] = -1

    top_indices = np.argsort(sims)[::-1][:top_k]

    results = []
    for rank, i in enumerate(top_indices, 1):
        results.append({
            'rank': rank,
            'node_id': node_ids[i],
            'type': nodes_df.iloc[i]['type'],
            'title': nodes_df.iloc[i]['title'],
            'similarity': sims[i],
        })

    return pd.DataFrame(results)

In [ ]:
# Query: Article 6 (High-Risk AI Systems)
print('═' * 70)
print('Query: Article 6 — Classification Rules for High-Risk AI Systems')
print('═' * 70)
find_nearest_neighbours('article_6', gnn_embeddings, nodes_df, top_k=10)

In [ ]:
# Query: Article 5 (Prohibited AI Systems)
print('═' * 70)
print('Query: Article 5 — Prohibited AI Practices')
print('═' * 70)
find_nearest_neighbours('article_5', gnn_embeddings, nodes_df, top_k=10)

In [ ]:
# Query: Definition 1 (AI System)
print('═' * 70)
print('Query: Definition — AI System')
print('═' * 70)
find_nearest_neighbours('definition_1', gnn_embeddings, nodes_df, top_k=10)

---
## 9. Baseline Comparison

Compare GNN embeddings vs the raw sentence-transformer embeddings (no graph structure).

In [ ]:
# Load raw text embeddings (without GNN)
raw_features = np.load(f"../{config['paths']['node_features']}")
# Only use the text embedding part (first 384 dims)
raw_text_emb = raw_features[:, :384]
# Normalise
raw_text_emb = raw_text_emb / (np.linalg.norm(raw_text_emb, axis=1, keepdims=True) + 1e-8)

print(f'Raw text embeddings shape:  {raw_text_emb.shape}')
print(f'GNN embeddings shape:       {gnn_embeddings.shape}')

In [ ]:
# Compare: which of Article 6's graph neighbours are ranked higher by GNN vs raw?
query = 'article_6'

print('── GNN Embeddings: Top 10 for Article 6 ──')
gnn_nn = find_nearest_neighbours(query, gnn_embeddings, nodes_df, top_k=10)
print(gnn_nn.to_string(index=False))

print('\n── Raw Text Embeddings: Top 10 for Article 6 ──')
raw_nn = find_nearest_neighbours(query, raw_text_emb, nodes_df, top_k=10)
print(raw_nn.to_string(index=False))

In [ ]:
# Quantitative comparison: average similarity to graph neighbours

def avg_neighbour_similarity(embeddings, edge_index, num_nodes, sample_n=500):
    """Average cosine similarity between connected nodes."""
    n_edges = edge_index.size(1)
    sample_idx = torch.randperm(n_edges)[:min(sample_n, n_edges)]

    sims = []
    for idx in sample_idx:
        src = edge_index[0, idx].item()
        tgt = edge_index[1, idx].item()
        sim = cosine_similarity(
            embeddings[src:src+1], embeddings[tgt:tgt+1]
        )[0, 0]
        sims.append(sim)
    return np.mean(sims), np.std(sims)

gnn_mean, gnn_std = avg_neighbour_similarity(gnn_embeddings, data.edge_index, data.num_nodes)
raw_mean, raw_std = avg_neighbour_similarity(raw_text_emb, data.edge_index, data.num_nodes)

print(f'Average cosine similarity between connected nodes:')
print(f'  GNN embeddings:       {gnn_mean:.4f} ± {gnn_std:.4f}')
print(f'  Raw text embeddings:  {raw_mean:.4f} ± {raw_std:.4f}')
print(f'  Improvement:          {gnn_mean - raw_mean:+.4f}')

# Bar chart
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    ['Raw Text\n(Sentence-Transformer)', 'GNN\n(GraphSAGE)'],
    [raw_mean, gnn_mean],
    yerr=[raw_std, gnn_std],
    color=['#FF9800', '#4CAF50'],
    capsize=5, width=0.5
)
ax.set_ylabel('Mean Cosine Similarity')
ax.set_title('Neighbour Similarity: Raw vs GNN Embeddings')
for bar, val in zip(bars, [raw_mean, gnn_mean]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontweight='bold')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

---
## 10. Save Final Embeddings

In [ ]:
# Save GNN embeddings
emb_path = Path(f"../{config['paths']['final_embeddings']}")
emb_path.parent.mkdir(parents=True, exist_ok=True)
np.save(str(emb_path), gnn_embeddings)

print(f'✓ Saved GNN embeddings ({gnn_embeddings.shape}) to: {emb_path}')

---
## Summary

| Item | Value |
|------|-------|
| **Architecture** | GraphSAGE (2-layer, mean aggregation) |
| **Training** | Self-supervised contrastive (InfoNCE) |
| **Input features** | Sentence-Transformer embeddings (384d) + metadata (7d) = 391d |
| **Output embeddings** | 128-dimensional, L2-normalised |
| **Graph** | 974 nodes, 3873 edges, 4 edge types |

### Key Takeaways

1. **Convergence**: The model converges within the patience window, with loss curves showing smooth descent.
2. **Structure-aware embeddings**: t-SNE shows meaningful clustering — nodes of the same type and chapter group together.
3. **GNN vs Raw**: GNN embeddings produce higher cosine similarity between connected nodes, showing the model has learned to encode graph structure.
4. **Retrieval quality**: Nearest-neighbour queries return semantically and structurally related nodes, validating the embeddings for downstream retrieval.